## English Teacher Model

Read xlsx, create pandas dataframe, select definition of each tactics, embed by LLM, and save to csv

In [ ]:
import plotly.io as pio

pio.renderers.default = "vscode" # for vscode coding
#pio.renderers.default = "notebook" # for github.io rendering

In [2]:
tactics_file = "enterprise-attack-v19.1-tactics.xlsx"
techniques_file = "enterprise-attack-v19.1-techniques.xlsx"

tactics_embeddings_file = "tactics_embeddings.csv"
techniques_embeddings_file = "techniques_embeddings.csv"

llm_model = "nomic-ai/nomic-embed-text-v2-moe"

In [6]:
# test gpu, tpu
import torch
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [27]:
import pandas as pd
import numpy as np
tactics_df = pd.read_excel(tactics_file, engine='openpyxl')
tactics_df.sort_values(by='ID', inplace=True)

tactics_df.head()

In [28]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(llm_model, trust_remote_code=True, device = device)
tactics_df['embedding'] = tactics_df['description'].apply(lambda x: model.encode(x).tolist())
tactics_df.head()

In [ ]:

# umap visualization
import umap

umap_model = umap.UMAP(n_neighbors=5, n_components=3, metric='cosine', n_jobs=-1, random_state=42)
embedding_3d = umap_model.fit_transform(list(tactics_df['embedding']))

In [32]:
# plot
import plotly.express as px
import plotly.io as pio
import matplotlib.cm as cm

fig = px.scatter_3d(
    embedding_3d,
    x=0,
    y=1,
    z=2,
    hover_data={'name': tactics_df['name'], 'ID': tactics_df['ID']},
    color = list(range(len(tactics_df))),
)
fig.update_layout(title='Tactics Embeddings UMAP Visualization')
fig.show()

In [ ]:
# use tactics_df['embedding'] as the bases for subspace construction
teacher_subspace = tactics_df['embedding'].tolist()


In [ ]:
teacher_subspace.len()

## Expert Model